# Stage 5 — валидация эксперимента

Notebook визуализирует только артефакты, рассчитанные `src.experiments.validation` на реальной feature mart. Он не генерирует демонстрационные результаты.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

ARTIFACT_DIR = Path('../data/processed/experiment_validation')
required = {
    'srm': ARTIFACT_DIR / 'srm.parquet',
    'aa_summary': ARTIFACT_DIR / 'aa_summary.parquet',
    'aa': ARTIFACT_DIR / 'aa_simulations.parquet',
    'bootstrap_summary': ARTIFACT_DIR / 'bootstrap_summary.parquet',
    'bootstrap': ARTIFACT_DIR / 'bootstrap_distribution.parquet',
    'segments': ARTIFACT_DIR / 'segment_analysis.parquet',
}
missing = [str(path) for path in required.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        'Нет артефактов Stage 5. Сначала выполните python -m src.experiments.validation. ' + str(missing)
    )

## SRM и сводные показатели

In [ ]:
display(pl.read_parquet(required['srm']).to_pandas())
display(pl.read_parquet(required['aa_summary']).to_pandas())
display(pl.read_parquet(required['bootstrap_summary']).to_pandas())

## Распределение A/A p-value

In [ ]:
aa = pl.read_parquet(required['aa']).to_pandas()
sns.histplot(aa, x='p_value', bins=20)
plt.axhline(len(aa) / 20, color='black', linestyle='--', label='Ожидание Uniform')
plt.title('A/A: распределение p-value')
plt.legend()
plt.show()

## Bootstrap-распределение uplift

In [ ]:
bootstrap = pl.read_parquet(required['bootstrap']).to_pandas()
sns.histplot(bootstrap, x='absolute_uplift', bins=40, kde=True)
plt.axvline(0, color='black', linestyle='--')
plt.title('Bootstrap uncertainty absolute uplift')
plt.show()

## Сегменты с поправкой Benjamini–Hochberg

In [ ]:
segments = pl.read_parquet(required['segments']).sort(
    ['dimension', 'adjusted_p_value']
).to_pandas()
display(segments)
g = sns.catplot(
    data=segments, x='absolute_uplift', y='segment', col='dimension',
    col_wrap=2, kind='bar', sharey=False, height=4
)
g.set_titles('{col_name}')
g.fig.suptitle('Absolute uplift по сегментам', y=1.02)
plt.show()